In [ ]:
# @title Import necessary Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from textwrap import fill
import pickle

In [ ]:
# @title Default the function to plot spider(radar) charts for multidimensional comparison
def plot_spider(df,trader,title):

  categories = df.columns.tolist()
  N = len(categories)
  angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
  angles += angles[:1]
  fig, ax = plt.subplots(constrained_layout=True,subplot_kw=dict(projection='polar'))
  for idx, row in df.iterrows():
    values = row.tolist()
    values += values[:1]          # close loop
    ax.plot(angles, values, linewidth=2, label=idx)
    ax.fill(angles, values, alpha=0.15)
  ax.set_xticks(angles[:-1])
  ax.set_xticklabels(categories)
  ax.set_title(title,pad=20)
  ax.grid(True)
  ax.legend(loc='upper right', bbox_to_anchor=(1.5, 1))
  plt.savefig(title+".jpg",bbox_inches='tight')
  plt.show()



In [ ]:
# @title Defining Function to display dataframes as a matplotlib table
def plot_data_table(df,title):

    df=round(df,2)
    df.index.name="Sentiment"
    fig, ax = plt.subplots(figsize=(10, 2 + 0.3*len(df)))
    ax.axis('off')
    tbl = ax.table(
    cellText=df.reset_index().values,
    colLabels=df.reset_index().columns,
    loc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(6)
    plt.title(title, pad=20)
    plt.show()


In [ ]:
fear_and_greed_df=pd.read_csv("fear_greed_index.csv",index_col="date")
fear_and_greed_df["timestamp"]=pd.to_datetime(fear_and_greed_df["timestamp"])
fear_and_greed_df.drop(columns=["timestamp"],inplace=True)
fear_and_greed_df.index=pd.to_datetime(fear_and_greed_df.index)
sentiment_list=fear_and_greed_df['classification'].value_counts().index.to_list()
historical_data_df=pd.read_csv("historical_data.csv")

In [ ]:
# @title Normalizing sentiment scores with average of each category
color_map = {
    'Extreme Fear': 'red',
    'Fear': 'orange',
    'Neutral': 'yellow',
    'Greed': 'green',
    'Extreme Greed': 'darkgreen'
}
class_means = fear_and_greed_df.groupby('classification')['value'].mean()
fear_and_greed_df_plot=fear_and_greed_df.copy()
fear_and_greed_df_plot['normalized'] = fear_and_greed_df_plot['value'] / fear_and_greed_df_plot['classification'].map(class_means)
colors = fear_and_greed_df_plot['classification'].map(color_map)
plt.figure(figsize=(10,4))
plt.bar(fear_and_greed_df_plot.index, fear_and_greed_df_plot['normalized'], color=colors)
plt.xlabel('Date')
plt.ylabel('Sentiment Value')
plt.title('Sentiment Over Time')
legend_labels = [Patch(color=c, label=l) for l, c in color_map.items()]
plt.legend(handles=legend_labels, title='Sentiment'
,loc='upper left', bbox_to_anchor=(1.05,1.0))
plt.tight_layout()
plt.savefig("Normalized Market Sentiment Scores over Time")
plt.show()

Autocorrelation testing for sentment scores

Step-1: Stationarity Check

In [ ]:
series = fear_and_greed_df_plot['value']

# Stationarity check
from statsmodels.tsa.stattools import adfuller
res = adfuller(series)
print('ADF p-value:', res[1])


# Autocorrelation tests
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.stattools import durbin_watson
import matplotlib.pyplot as plt

plot_acf(series, lags=30)
plt.savefig("ACF Plot for Sentiment Scores")
plt.show()

print(acorr_ljungbox(series, lags=[10], return_df=True))
print("Durbin-Watson:", durbin_watson(series))


In [ ]:
# @title Defining funtion to calculate Hurst Exponent for a time series
def hurst_robust(ts):
    ts = np.array(ts)
    lags = range(2, min(100, len(ts)//2))
    tau = [np.sqrt(np.mean(np.square(ts[lag:] - ts[:-lag]))) for lag in lags]
    poly = np.polyfit(np.log(lags), np.log(tau), 1)
    return 2.0 * poly[0]

h = hurst_robust(series)
print("Hurst =", h)

In [ ]:
# @title Combining Trades Data With Fear Data
accounts_list=historical_data_df['Account'].value_counts().index.to_list()
trader_list={}
for v in range(len(accounts_list)):
  trader_list[accounts_list[v]]="Trader "+str(v+1)
df_tl = pd.DataFrame.from_dict(trader_list, orient='index', columns=["Name"])
df_tl.index.name="Account Address"
df_tl.to_excel("Account Mapping.xlsx")
historical_data_df['Account']=historical_data_df['Account'].map(trader_list)
historical_data_df["date"]=pd.to_datetime(historical_data_df["Timestamp IST"],format='%d-%m-%Y %H:%M').dt.date
fear_and_greed_df_v1=fear_and_greed_df.copy()
fear_and_greed_df_v1.reset_index(inplace=True)
fear_and_greed_df_v1['date']=pd.to_datetime(fear_and_greed_df_v1['date']).dt.date
combined_dataset=historical_data_df.merge(fear_and_greed_df_v1,left_on='date',right_on='date',how='left')
combined_dataset.to_excel("Combined Trade and Fear Dataset.xlsx")
coins_list=combined_dataset['Coin'].value_counts().index.to_list()


In [ ]:
# @title Distribution of Trades by Transaction Types
direcction_table=pd.DataFrame(combined_dataset['Direction'].value_counts())
top_transactons=list(combined_dataset['Direction'].value_counts().index)[:4]
total_transactions=direcction_table['count'].values.sum()
total_transactions_top=direcction_table[direcction_table.index.isin(top_transactons)]['count'].values.sum()
transaction_proportion=total_transactions_top/total_transactions
((direcction_table/total_transactions)*100).plot(kind='bar')
plt.ylabel("Transaction Proportion")
plt.xlabel("Tansaction Types")
plt.title(fill("Total proportion of transactions accounted for by {}, {}, {}, {} is {}%".format(
            top_transactons[0],top_transactons[1],top_transactons[2],top_transactons[3],
            round(transaction_proportion*100)), width=55))
plt.savefig("Transaction types overall.jpg",bbox_inches='tight')


In [ ]:
# @title Number of Orders by Direction in Different regimes of market sentiments
dirctional_transactions=combined_dataset[combined_dataset['Direction'].isin(top_transactons)]
summary13=dirctional_transactions.groupby(["classification","Direction"]).agg({'Size USD':['count']}).unstack()
summary13.columns=[vol for vol1,vol2,vol in summary13.columns]
summary13.plot(kind='bar',xlabel="Market Sentiment Type",ylabel="Number of Orders",
              title="Number of Orders by Direction in Different regimes of market sentiments",
              legend=True)
plt.savefig("Number of Orders by Direction in Different regimes of market sentiments.jpg",bbox_inches='tight')


In [ ]:
# @title Average Closed PnL by Direction in Different regimes of market sentiments
direcction_transactions_v2=[[top_transactons[1],top_transactons[3]]]
dirctional_transactions_v3=combined_dataset[combined_dataset['Direction'].isin(['Close Long', 'Close Short'])]
summary14=dirctional_transactions_v3.groupby(["classification","Direction"]).agg({'Closed PnL':['mean']}).unstack()
summary14.columns=[vol for vol1,vol2,vol in summary14.columns]
summary14.plot(kind='bar',xlabel="Market Sentiment Type",ylabel="Average Closed PnL",
              title="Average Closed PnL by Direction in Different regimes of market sentiments",
              legend=True)
plt.savefig("Average Closed PnL by Direction in Different regimes of market sentiments.jpg",bbox_inches='tight')

In [ ]:
dirctional_transactions_v4=combined_dataset[combined_dataset['Direction'].isin(top_transactons)]
summary15=dirctional_transactions_v4.groupby(["classification","Direction"]).agg({'Size USD':['mean']}).unstack()
summary15.columns=[vol for vol1,vol2,vol in summary15.columns]
summary15.plot(kind='bar',xlabel="Market Sentiment Type",ylabel="Average Position Size",
              title="Average Position Size by Direction in Different regimes of market sentiments",
              legend=True)
plt.savefig("Average Position Size by Direction in Different regimes of market sentiments.jpg",bbox_inches='tight')

trade_data is a nested dictonery containing the coinwise trades data as values against the trader as a key

In [ ]:
trade_data={}
for i in range(len(trader_list.values())):
  coin_trade={}
  account_trade=combined_dataset[combined_dataset['Account']==list(trader_list.values())[i]]
  coins_traded=account_trade['Coin'].value_counts().index.to_list()
  for j in range(len(coins_traded)):
    coin_trade[coins_traded[j]]=account_trade[account_trade['Coin']==coins_traded[j]]
  trade_data[list(trader_list.values())[i]]=coin_trade
with open("Coin wise Trade Data for Individual Traders.pkl", "wb") as f:
    pickle.dump(trade_data, f)

In [ ]:
# @title Function to get coinwise trades for individual traders
def get_coinwise_trades(trade_data,trader):
  filter1=trade_data[trader]
  for coin in filter1.keys():
    df=filter1[coin]
    fig, ax = plt.subplots(figsize=(10, 2 + 0.3*len(df)))
    ax.axis('off')
    tbl = ax.table(cellText=df.values, colLabels=df.columns, loc='center')
    plt.title("Trades for {} by {}".format(coin,trader), pad=20)
    plt.show()

anaytics_data is a nested dictonery containing the coinwise trade analytics data as values against the trader as a key

In [ ]:
# @title analytics_data is a nested dictonery containing the coinwise trades data as values against the trader as a key
analystics_data={}
for k in list(trade_data.keys()):
  coin_dict=trade_data[k]
  dict_1={}
  for h in coin_dict.keys():
    x1=coin_dict[h]
    x1.index=x1['Timestamp IST']
    x3=x1.groupby(['classification']).agg({'Side':['count'],'Closed PnL':['sum','mean'],'Fee':['sum','mean'],
                                       'value':['mean'],'Size Tokens':['sum','mean'],'Size USD':['sum','mean']})
    list1=[f"{t1} {t2}".title() for t2, t1 in x3.columns]
    list1[0]='Number of Trades'
    x3.columns=list1
    x3.index.name="Sentiment Regime"
    dict_1[h]=x3
  analystics_data[k]=dict_1
with open("Coin wise Trade Analytics for Individual Traders.pkl", "wb") as f:
    pickle.dump(analystics_data, f)

In [ ]:
# @title Function to get trade analytics by coin by trader
def get_trader_analytics_by_coin(analystics_data,trader):
  filter1=analystics_data[trader]
  for coin in filter1.keys():
    title="Trade Analytics for {}".format(coin)
    plot_data_table(filter1[coin],title)

In [ ]:
# @title Compilation of Overall Trade Analytics for Each Trader
overall_trader_analytics={}
for i in range(len(list(trader_list.values()))):
  total_trades_v1=combined_dataset[combined_dataset['Account']==list(trader_list.values())[i]]
  total_trades_v2=total_trades_v1.groupby(['classification']).agg({'Side':['count'],'Closed PnL':['sum','mean'],'Fee':['sum','mean'],
                                       'value':['mean'],'Size Tokens':['sum','mean'],'Size USD':['sum','mean']})
  total_trades_v2_list=["{} {}".format(t1,t2).title() for t2,t1 in total_trades_v2.columns]
  total_trades_v2_list[0]="Number of Trades"
  total_trades_v2.columns=total_trades_v2_list
  overall_trader_analytics[list(trader_list.values())[i]]=total_trades_v2
with open("Overall Trade Analytics for Each Trader.pkl", "wb") as f:
    pickle.dump(overall_trader_analytics, f)

In [ ]:
# @title Function for Overall Trade Analytics for Each Trader
def get_overall_trader_analytics(overall_trader_analytics,trader):
  title="Overall Trade Analytics"
  plot_data_table(overall_trader_analytics[trader],title)

In [ ]:
combined_dataset.groupby(["classification"]).agg({'value':['mean']}).plot(kind='bar',
                                                                          xlabel="Market Sentiment Type",
                                                                          ylabel="Rating",
                                                                          title="Average Rating of Market Sentiments",
                                                                          legend=False)
plt.savefig("Average Reting of Market Sentiments.jpg")

In [ ]:
combined_dataset.groupby(["classification"]).agg({'Closed PnL':['mean']}).plot(kind='bar',
                                                                          xlabel="Market Sentiment Type",
                                                                          ylabel="Average Closed PnL",
                                                                          title="Average Closed PnL in different regmes of Market Sentiments",
                                                                          legend=False)
plt.savefig("Average Closed PnL in different regmes of Market Sentiments.jpg",bbox_inches='tight')

In [ ]:
combined_dataset.groupby(["classification"]).agg({'Closed PnL':['sum']}).plot(kind='bar',
                                                                          xlabel="Market Sentiment Type",
                                                                          ylabel="Total Closed PnL",
                                                                          title="Total Closed PnL of in different regimes of Market Sentiments",
                                                                          legend=False)
plt.savefig("Total Closed PnL of in different regimes of Market Sentiments.jpg",bbox_inches='tight')

In [ ]:
summary1=combined_dataset.groupby(["classification","Side"]).agg({'Side':['count']}).unstack()
summary1.columns=[vol for vol1,vol2,vol in summary1.columns]
summary1.plot(kind='bar',xlabel="Market Sentiment Type",ylabel="Count of Orders Placed",
              color=['red','blue'],title="Count of Orders Placed in different Market Sentiment Regimes",
              legend=True)
plt.savefig("Count of Orders Placed in different Market Sentiment Regimes.jpg",bbox_inches='tight')

In [ ]:
combined_dataset[combined_dataset['Closed PnL']<0].groupby(["classification"]).agg({'Closed PnL':['sum']}).plot(kind='bar',
                                                                          xlabel="Market Sentiment Type",
                                                                          ylabel="Total Negative Closed PnL",
                                                                          title="Total Negative Closed PnL in different regimes of Market Sentiments",
                                                                          legend=False)
plt.savefig("Total Negative Closed PnL in different regimes of Market Sentiments.jpg",bbox_inches='tight')

In [ ]:
combined_dataset[combined_dataset['Closed PnL']<0].groupby(["classification"]).agg({'Closed PnL':['mean']}).plot(kind='bar',
                                                                          xlabel="Market Sentiment Type",
                                                                          ylabel="Average Negative Closed PnL",
                                                                          title="Average Negative Closed PnL in different regimes of Market Sentiments",
                                                                          legend=False)
plt.savefig("Average Negative Closed PnL in different regimes of Market Sentiments.jpg",bbox_inches='tight')

In [ ]:
combined_dataset[combined_dataset['Closed PnL']>0].groupby(["classification"]).agg({'Closed PnL':['sum']}).plot(kind='bar',
                                                                          xlabel="Market Sentiment Type",
                                                                          ylabel="Total Positive Closed PnL",
                                                                          title="Total Positive Closed PnL in different regimes of Market Sentiments",
                                                                          legend=False)
plt.savefig("Total Positive Closed PnL in different regimes of Market Sentiments.jpg",bbox_inches='tight')

In [ ]:
combined_dataset[combined_dataset['Closed PnL']>0].groupby(["classification"]).agg({'Closed PnL':['mean']}).plot(kind='bar',
                                                                          xlabel="Market Sentiment Type",
                                                                          ylabel="Average Positive Closed PnL",
                                                                          title="Average Positive Closed PnL in different regimes of Market Sentiments",
                                                                          legend=False)
plt.savefig("Average Positive  Closed PnL in different regimes of Market Sentiments.jpg",bbox_inches='tight')

In [ ]:
summary2=combined_dataset.groupby(["classification","Side"]).agg({'Fee':['mean']}).unstack()
summary2.columns=[vol for vol1,vol2,vol in summary2.columns]
summary2.plot(kind='bar',xlabel="Market Sentiment Type",ylabel="Average Fee of Orders Placed",
              color=['red','blue'],title="Average Fees of Orders Placed in different regimes of Market Sentiments",
              legend=True)
plt.savefig("Average Fees of Orders Placed in different regimes of Market Sentiments.jpg",bbox_inches='tight')

In [ ]:
summary2x=combined_dataset.groupby(["classification","Side"]).agg({'Size USD':['mean']}).unstack()
summary2x.columns=[vol for vol1,vol2,vol in summary2x.columns]
summary2x.plot(kind='bar',xlabel="Market Sentiment Type",ylabel="Average Fee of Orders Placed",
              color=['red','blue'],title="Average Size of Orders Placed in different regimes of Market Sentiments",
              legend=True)
plt.savefig("Average Size of Orders Placed in different regimes of Market Sentiments.jpg",bbox_inches='tight')

In [ ]:
summary3=combined_dataset.groupby(["Coin"]).agg({'Closed PnL':['mean']})
summary3.sort_values(by=summary3.columns[0],ascending=True)[:10].plot(kind='bar',
                                                                          xlabel="Coin",
                                                                          ylabel="Average Closed PnL",
                                                                          title="Average Closed PnL of 10 least Profitable Coins",
                                                                          legend=False)

In [ ]:
summary3.sort_values(by=summary3.columns[0],ascending=False)[:10].plot(kind='bar',
                                                                          xlabel="Coin",
                                                                          ylabel="Average Closed PnL",
                                                                          title="Average Closed PnL of 10 most Profitable Coins",
                                                                          legend=False)

In [ ]:
summary4=combined_dataset.groupby(["Coin"]).agg({'Closed PnL':['sum']})
summary4.sort_values(by=summary4.columns[0],ascending=True)[:10].plot(kind='bar',
                                                                          xlabel="Coin",
                                                                          ylabel="Total Closed PnL",
                                                                          title="Total Closed PnL of 10 least Profitable Coins",
                                                                          legend=False)

In [ ]:
summary4.sort_values(by=summary4.columns[0],ascending=False)[:10].plot(kind='bar',
                                                                          xlabel="Coin",
                                                                          ylabel="Total Closed PnL",
                                                                          title="Total Closed PnL of 10 least Profitable Coins",
                                                                          legend=False)

In [ ]:
summary5=combined_dataset.groupby(["Coin"]).agg({'Closed PnL':['count']})
summary5.sort_values(by=summary5.columns[0],ascending=True)[:10].plot(kind='bar',
                                                                          xlabel="Coin",
                                                                          ylabel="Total Trades",
                                                                          title="Total Trades of 10 least Traded Coins",
                                                                          legend=False)

In [ ]:
summary5.sort_values(by=summary5.columns[0],ascending=False)[:10].plot(kind='bar',
                                                                          xlabel="Coin",
                                                                          ylabel="Total Trades",
                                                                          title="Total Trades of 10 most Traded Coins",
                                                                          legend=False)

In [ ]:
summary6=combined_dataset[combined_dataset['classification']=='Fear'].groupby(["Account"]).agg({'Closed PnL':['mean']})
summary6.sort_values(by=summary6.columns[0],ascending=True).plot(kind='bar',
                                                                          xlabel="Coin",
                                                                          ylabel="Total Closed PnL",
                                                                          title="Total Closed PnL of 10 least Profitable Coins",
                                                                          legend=False)

In [ ]:
summary10=combined_dataset.groupby(["Coin"]).agg({'Fee':['count','mean']})
summary10.columns=['Trades','Average Fees']
summary10["Percentage_trades"]=(summary10['Trades']/summary10['Trades'].sum())*100
summary10_top_10=summary10.sort_values(by="Percentage_trades",ascending=False)[:10]
total=round(summary10_top_10["Percentage_trades"].sum(),2)
summary10_top_10["Percentage_trades"].plot.pie(ylabel=" ",
    labels=[f'{i} = {round(v,2)}%' for i, v in zip(summary10_top_10.index, summary10_top_10["Percentage_trades"])])
plt.title("Total Contribution of Top 10 Traded Coins in Total Number of Trades = {}%".format(total))
plt.savefig("Top10 Overall.jpg",bbox_inches='tight')

In [ ]:
summary9=combined_dataset.groupby(["Coin","Side"]).agg({'Fee':['count','mean']})
summary9.columns=['Trades','Average Fees']
summary9=summary9.xs('BUY', level='Side')
summary9["Percentage_trades"]=(summary9['Trades']/summary9['Trades'].sum())*100
summary9_top_10=summary9.sort_values(by="Percentage_trades",ascending=False)[:10]
total_buy=round(summary9_top_10["Percentage_trades"].sum(),2)
summary9_top_10["Percentage_trades"].plot.pie(ylabel=" ",
    labels=[f'{i} = {round(v,2)}%' for i, v in zip(summary9_top_10.index, summary9_top_10["Percentage_trades"])])
plt.title("Total Contribution of Top 10 Traded Coins in Total Number of Buy Trades = {}%".format(total_buy))
plt.savefig("Top10 Buy.jpg",bbox_inches='tight')


In [ ]:
summary11=combined_dataset.groupby(["Coin","Side"]).agg({'Fee':['count','mean']})
summary11.columns=['Trades','Average Fees']
summary11=summary11.xs('SELL', level='Side')
summary11["Percentage_trades"]=(summary11['Trades']/summary11['Trades'].sum())*100
summary11_top_10=summary11.sort_values(by="Percentage_trades",ascending=False)[:10]
total_sell=round(summary11_top_10["Percentage_trades"].sum(),2)
summary11_top_10["Percentage_trades"].plot.pie(ylabel=" ",
    labels=[f'{i} = {round(v,2)}%' for i, v in zip(summary11_top_10.index, summary11_top_10["Percentage_trades"])])
plt.title("Total Contribution of Top 10 Traded Coins in Total Number of Sell Trades = {}%".format(total_sell))
plt.savefig("Top10 Sell.jpg",bbox_inches='tight')


In [ ]:
regime_transaction_cluster={}
for sentiment in sentiment_list:
  summary12=combined_dataset[combined_dataset['classification']==sentiment].groupby(["Coin"]).agg({'Fee':['count','mean']})
  summary12.columns=['Trades','Average Fees']
  summary12["Percentage_trades"]=(summary12['Trades']/summary12['Trades'].sum())*100
  summary12_top_10=summary12.sort_values(by="Percentage_trades",ascending=False)[:10]
  total_sentiment=round(summary12_top_10["Percentage_trades"].sum(),2)
  regime_transaction_cluster[sentiment]=summary12_top_10
  plt.figure(figsize=(6,6))
  summary12_top_10["Percentage_trades"].plot.pie(ylabel=" ",
    labels=[f'{i} = {round(v,2)}%' for i, v in zip(summary12_top_10.index, summary12_top_10["Percentage_trades"])])
  title="Total Contribution of Top 10 Traded Coins in Total Number of Trades = {}% in {} Regime".format(total_sentiment,sentiment)
  plt.title(title)
  plt.savefig(title+".jpg",bbox_inches='tight')
with open("Coinwise Distribution of Transactions in Market Regimes.pkl", "wb") as f:
    pickle.dump(regime_transaction_cluster, f)

In [ ]:
regime_PnL_cluster={}
for sentiment in sentiment_list:
  summary12=combined_dataset[combined_dataset['classification']==sentiment].groupby(["Coin"]).agg({'Closed PnL':['sum']})
  summary12.columns=['Total PnL']
  summary12_top_10=summary12.sort_values(by='Total PnL',ascending=False)[:10]
  summary12_top_10["Percentage_Total PnL"]=(summary12_top_10['Total PnL']/summary12_top_10['Total PnL'].sum())*100
  regime_PnL_cluster[sentiment]=summary12_top_10
  plt.figure(figsize=(6,6))
  summary12_top_10["Percentage_Total PnL"].plot.pie(ylabel=" ",
    labels=[f'{i} = {round(v,2)}%' for i, v in zip(summary12_top_10.index, summary12_top_10["Percentage_Total PnL"])])
  title="Contribution of Top 10 Coins in Total Booked PnL {} Regime".format(sentiment)
  plt.title(title)
  plt.savefig(title+".jpg",bbox_inches='tight')
with open("Coinwise Distribution of total Closed PnL in Market Regimes.pkl", "wb") as f:
    pickle.dump(regime_PnL_cluster, f)

In [ ]:
regime_mean_PnL_cluster={}
for sentiment in sentiment_list:
  summary12=combined_dataset[combined_dataset['classification']==sentiment].groupby(["Coin"]).agg({'Closed PnL':['mean']})
  summary12.columns=['Mean PnL']
  summary12_top_10=summary12.sort_values(by='Mean PnL',ascending=False)[:10]
  summary12_top_10["Corrected_Total PnL"]=(summary12_top_10['Mean PnL'])*2
  regime_mean_PnL_cluster[sentiment]=summary12_top_10
  plt.figure(figsize=(6,6))
  summary12_top_10["Mean PnL"].plot(kind='bar')
  title="Comparison of Top 10 Coins with highest Mean Booked PnL {} Regime".format(sentiment)
  plt.title(title)
  plt.savefig(title+".jpg",bbox_inches='tight')
with open("Coinwise Distribution of mean Closed PnL in Market Regimes.pkl", "wb") as f:
    pickle.dump(regime_mean_PnL_cluster, f)

In [ ]:
# @title Function to plot spider plots to compare total coinwise PnL distribution in different market regimes with trader PnL in the same coins in the same regimes
def plot_sentimentwise_total_pnl_alignment(regime_mean_PnL_cluster,sentiment_list,trader):
  for sentiment in sentiment_list:
    coin_perf=combined_dataset[combined_dataset['Coin'].isin(regime_PnL_cluster[sentiment].index.to_list())].groupby(['Account','Coin']).agg({"Closed PnL":['sum']}).unstack('Account')
    coin_perf=coin_perf.fillna(0)
    coin_perf.columns=[c for a,b,c in coin_perf.columns]
    coin_perf=(coin_perf.div(coin_perf.sum(axis=0), axis=1))*100
    etemp=coin_perf.merge(regime_PnL_cluster[sentiment],left_index=True,right_index=True,how="left")
    try:
      etemp_1=etemp[[trader,etemp.columns[-1]]]
      plot_spider(etemp_1.T,trader,"Total PnL Alignment in {}".format(sentiment))
    except KeyError:
      pass


In [ ]:
# @title Function to plot spider plots to compare average coinwise PnL distribution in different market regimes with trader PnL in the same coins in the same regimes
def plot_sentimentwise_mean_pnl_alignment(regime_PnL_cluster,sentiment_list,trader):
  for sentiment in sentiment_list:
    coin_perf=combined_dataset[combined_dataset['Coin'].isin(regime_PnL_cluster[sentiment].index.to_list())].groupby(['Account','Coin']).agg({"Closed PnL":['mean']}).unstack('Account')
    coin_perf=coin_perf.fillna(0)
    coin_perf.columns=[c for a,b,c in coin_perf.columns]
    coin_perf=(coin_perf.div(coin_perf.sum(axis=0), axis=1))*100
    etemp=coin_perf.merge(regime_PnL_cluster[sentiment],left_index=True,right_index=True,how="left")
    try:
      etemp_1=etemp[[trader,etemp.columns[-1]]]
      plot_spider(etemp_1.T,trader,"Mean PnL Alignment in {}".format(sentiment))
    except KeyError:
      pass


In [ ]:
metrics=['count','sum','mean']
levels=[sentiment_list,metrics]
df_trader_perform=pd.DataFrame(
    index=list(trader_list.values()),
    columns=pd.MultiIndex.from_product(levels, names=['c1','c2'])
)
df_trader_mentality=pd.DataFrame(
    index=list(trader_list.values()),
    columns=sentiment_list
)
df_trader_tendency=pd.DataFrame(
    index=list(trader_list.values()),
    columns=sentiment_list
)
for sentiment in sentiment_list:
  summary7=combined_dataset[combined_dataset['classification']==sentiment].groupby(["Account"]).agg({'value':['mean']})
  df_trader_mentality[sentiment]=summary7
  summary8=combined_dataset[combined_dataset['classification']==sentiment].groupby(["Account"]).agg({'Size USD':['mean']})
  df_trader_tendency[sentiment]=summary8
  for metric in metrics:
    summary6=combined_dataset[combined_dataset['classification']==sentiment].groupby(["Account"]).agg({'Closed PnL':[metric]})
    df_trader_perform[(sentiment,metric)]=summary6
df_trader_mentality.to_excel("Average rating of sentiment ratings by trader by sentiment.xlsx")
df_trader_tendency.to_excel("Average position size of trades by trader by sentiment.xlsx")
df_trader_perform.to_excel("Metirces of Closed PnL of trades by trader by sentiment.xlsx")


In [ ]:
# @title Function to generate spider plot to compare average trade sizes in different market regimes with trades executed by trades in the same regime
position_size=combined_dataset.groupby(["classification"]).agg({'Size USD':['mean']})
position_size.columns=['Average Position Size (USD)']
position_size=position_size.T
df4=df_trader_tendency.fillna(0)
def plot_trade_size_comparison(position_size,df4,trader):
  df_print=position_size
  df_print.loc["Trader Position Size"]=df4.loc[trader]
  plot_spider(df_print,trader,"Average Trade Size Distribiton: {}".format(trader))

In [ ]:
# @title Function to save the Trade Size Comparison plots
def save_trader_position_alignment_plots(position_size,df4,trader_list):
  for trader in list(trader_list.values()):
    plot_trade_size_comparison(position_size,df4,trader)
save_trader_position_alignment_plots(position_size,df4,trader_list)

In [ ]:
# @title Function to comprare the percentage of trades executed by each trader in different market regimes with the percentage of PnL booked in the corrsponding regimes
df1=df_trader_perform.xs('count',level='c2',axis=1).fillna(0)
df1=df1.div(df1.sum(axis=1),axis=0)*100
df2=df_trader_perform.xs('sum',level='c2',axis=1).fillna(0)
df2=df2.div(df2.sum(axis=1),axis=0)*100
def plot_trades_vs_pnl(df1,df2,trader):
  df_print=pd.DataFrame(columns=df1.columns)
  df_print.loc["Trade Precentage"]=df1.loc[trader]
  df_print.loc["PnL Precentage"]=df2.loc[trader]
  plot_spider(df_print,trader,"Trade Concentration vs PnL Conentration: {}".format(trader))

In [ ]:
# @title Function to save Trade and PnL concentration plots
def save_trader_position_pnl_alignment_plots(df1,df2,trader_list):
  for trader in list(trader_list.values()):
    plot_trades_vs_pnl(df1,df2,trader)
save_trader_position_pnl_alignment_plots(df1,df2,trader_list)

In [ ]:
# @title Function to comprare the average sentiment scores by sentiment with the average sentiment scores of the days when the traders executed their trades
sentiment_ratings=combined_dataset.groupby(["classification"]).agg({'value':['mean']})
sentiment_ratings.columns=['Average Sentiment Rating']
sentiment_ratings=sentiment_ratings.T
df5=df_trader_mentality.fillna(0)
def plot_sentimentwise_alignment(df5,trader):
  df_print=sentiment_ratings.copy()
  df_print.loc["Trader Rating"]=df5.loc[trader]
  df_print.fillna(0,inplace=True)
  plot_spider(df_print,trader,"BTC Sentiment Alignment: {}".format(trader))

In [ ]:
# @title Function to save sentiment alignment plots
def save_trader_sentiment_alignment_plots(df5,trader_list):
  for trader in list(trader_list.values()):
    plot_sentimentwise_alignment(df5,trader)
save_trader_sentiment_alignment_plots(df5,trader_list)

In [ ]:
# @title Generating Rank datarame for traders
rank_df0=combined_dataset[combined_dataset['Side']=="SELL"]
rank_df0['Transaction Outcome']=np.where(rank_df0['Closed PnL']>0,1,0)
rank_df1=rank_df0.groupby(['Account']).agg({'Closed PnL':['sum','mean'],'Fee':['sum','mean'],'Transaction Outcome':['mean','count']})
rank_df1.columns=["{} {}".format(t1,t2).title() for t2,t1 in rank_df1.columns]
for column in rank_df1.columns:
  rank_df1[column+" rank"]=rank_df1[column].rank(ascending=False)
rank_df1.to_excel("Rank Dataframe for Traders.xlsx")


In [ ]:
# @title Function to plot trader rank as matplotlib table
def plot_performance_rank(rank_df1,trader):
  temp_df1=pd.DataFrame(columns=rank_df1.columns)
  temp_df1.loc[trader]=rank_df1.loc[trader]
  temp_df1_rank=temp_df1[temp_df1.columns[-5:]].copy()
  title_2="Overall Rank"
  plot_data_table(temp_df1_rank,title_2)

In [ ]:
# @title Function to plot trader performance as matplotlib table
def plot_performance_overview(rank_df1,trader):
  temp_df1=pd.DataFrame(columns=rank_df1.columns)
  temp_df1.loc[trader]=rank_df1.loc[trader]
  temp_df1_figures=temp_df1[temp_df1.columns[:5]].copy()
  title_1="Overall Metrics"
  plot_data_table(temp_df1_figures,title_1)

In [ ]:
# @title Compiling trader performance in different sentiment regimes
rank_sentiment={}
for sentiment in sentiment_list:
  rank_df2=rank_df0[rank_df0['classification']==sentiment]
  rank_df3=rank_df2.groupby(['Account']).agg({'Closed PnL':['sum','mean'],'Fee':['sum','mean'],'Transaction Outcome':['mean','count']})
  rank_df3.columns=["{} {}".format(t1,t2).title() for t2,t1 in rank_df3.columns]
  for column in rank_df3.columns:
    rank_df3[column+" rank"]=rank_df3[column].rank(ascending=False)
  rank_sentiment[sentiment]=rank_df3
with open("Sentimentwise oerformance of Traders.pkl", "wb") as f:
    pickle.dump(rank_sentiment, f)


In [ ]:
# @title Saving Results of Sentiment wise ranks
for sentiment in rank_sentiment.keys():
  rank_sentiment[sentiment].to_excel("{} Rank.xlsx".format(sentiment))

In [ ]:
# @title Compiling data for sentimentwise ranking of trader performance
trader_analytics_concrete={}
for trader in list(trader_list.values()):
  ta=pd.DataFrame(columns=rank_df1[rank_df1.columns[-5:]].columns)
  ta.loc['Overall']=rank_df1[rank_df1.columns[-5:]].loc[trader]
  for sentiment in sentiment_list:
    try:
      rank_df2f=rank_sentiment[sentiment][rank_sentiment[sentiment].columns[-5:]]
      ta.loc[sentiment]=rank_df2f.loc[trader]
    except KeyError:
      pass
  trader_analytics_concrete[trader]=ta
with open("Sentimentwise ranking of trader performance.pkl", "wb") as f:
    pickle.dump(trader_analytics_concrete, f)

In [ ]:
# @title Function for getting the ranks of traders in differnt metrics in different regimes of market sentiment
def plot_sentimentwise_trader_rank(trader_analytics_concrete,trader):
  df=trader_analytics_concrete[trader]
  plt.figure()
  df.T.plot(kind='bar',xlabel="Metric")
  plt.title("Performance Summary: {}".format(trader),pad=20)
  plt.legend(loc=(1.03,0.6))

In [ ]:
# @title Function to save sentiment-wise trader ranks
def save_trader_performance_plots(trader_analytics_concrete,trader_list):
  for trader in list(trader_list.values()):
    plot_sentimentwise_trader_rank(trader_analytics_concrete,trader)
    plt.savefig("Performance Summary: {}.jpg".format(trader),bbox_inches='tight')
save_trader_performance_plots(trader_analytics_concrete,trader_list)

In [ ]:
# @title Function to plot combined indicators of sentiment analysis for traders
def plot_combined_trader_analytics(trader):
  fig1, ax = plt.subplots(2, 2, figsize=(16,16))
  img1 = plt.imread("Performance Summary: {}.jpg".format(trader))
  ax[0,0].imshow(img1)
  ax[0,0].set_title("Figure: 1",fontsize=14,fontweight='bold')

  img2 = plt.imread("BTC Sentiment Alignment: {}.jpg".format(trader))
  ax[0,1].imshow(img2)
  ax[0,1].set_title("Figure: 2",fontsize=14,fontweight='bold')

  img3 = plt.imread("Trade Concentration vs PnL Conentration: {}.jpg".format(trader))
  ax[1,0].imshow(img3)
  ax[1,0].set_title("Figure: 3",fontsize=14,fontweight='bold')

  img4 = plt.imread("Average Trade Size Distribiton: {}.jpg".format(trader))
  ax[1,1].imshow(img4)
  ax[1,1].set_title("Figure: 4",fontsize=14,fontweight='bold')

  for a in ax.flatten():
     a.axis('off')

  account=[k for k, v in trader_list.items() if v == trader][0]
  fig1.suptitle(f"Combined BTC Market Sentiment Analysis for {trader} \n Account: {account}", fontsize=18,fontweight='bold',y=1.02)
  plt.tight_layout()
  plt.savefig("Combined_Sentiment_analysis{}.jpg".format(trader),bbox_inches='tight')

In [ ]:
# @title Function to save combined analysis plots for traders
def save_combined_trader_analytics(trader_list):
  for trader in trader_list.values():
    plot_combined_trader_analytics(trader)
save_combined_trader_analytics(trader_list)

In [ ]:
# @title Function to generate consolidated trader performance report
def get_account_details(trader,get_trades=False):
  print("Trader Performance Report for {}".format(trader))
  print("Account ID : {}".format([k for k, v in trader_list.items() if v == trader][0]))
  print("This section summarizes the performance metrics of the trader.")
  plot_performance_overview(rank_df1,trader)
  print("This section summarizes the ranks of the trader according to the performance metrics.")
  plot_performance_rank(rank_df1,trader)
  print("This section summarizes the ranks of the trader according to the performance in different market segments.")
  plot_sentimentwise_trader_rank(trader_analytics_concrete,trader)
  print("This section summarizes the trade metrics by sentiment regimes.")
  get_overall_trader_analytics(overall_trader_analytics,trader)
  print("This section identifies if the portfolio of the trader is aligned with the top perfoming coins in each market regime.")
  plot_sentimentwise_alignment(df5,trader)
  print("This section shows if the trades and closed PnL are aligned in each market sentiment.")
  plot_trades_vs_pnl(df1,df2,trader)
  print("This section shows if the distribution of total closed PnL in each market sentiment is aligned with the top performing coins in the same market sentiment.")
  plot_sentimentwise_total_pnl_alignment(regime_PnL_cluster,sentiment_list,trader)
  print("This section shows how the average trade size in each market sentiment compares with that of the trader.")
  plot_trade_size_comparison(position_size,df4,trader)
  print("This section compares the mean PnL in each during each market sentiment regime with that of the trader in the same regime")
  plot_sentimentwise_mean_pnl_alignment(regime_mean_PnL_cluster,sentiment_list,trader)
  print("This section summarizes the trade metrics of by coin.")
  get_trader_analytics_by_coin(analystics_data,trader)

  if get_trades:
    get_coinwise_trades(trade_data,trader)
  else:
    pass

In [ ]:
# @title Example use case of trader performance report generation
get_account_details('Trader 1')

In [ ]:
# @title Download all generated outputs from the code
!zip -r all_outputs.zip /content
from google.colab import files
files.download('all_outputs.zip')